# 6.11 — Weight Initialization

Weight initialization chooses the starting scale of neural-network weights so activations and gradients can travel through many layers without fading to zero or exploding. In this lesson, you will build variance-preserving initialization from first principles in NumPy, inspect why Xavier, He, orthogonal, and LSUV initializers exist, and connect the formulas to the practical training failures they prevent.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build weight initialization one idea at a time. Run each cell in order and read the printed intermediate values — every piece of math is visible, including the forward variance and the backward-gradient scale. This walkthrough is self-contained and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, random weights, and numerical checks for initialization.
import matplotlib.pyplot as plt  # compact visualizations for signal and gradient scale.
np.random.seed(0)  # reproducibility for all scratch experiments.

### 1. Why random weight scale matters

A dense layer computes $z=xW+b$. If the weights start too tiny, every layer shrinks the signal; if they start too large, each layer magnifies the signal. The whole point of initialization is not to make weights "random" in a vague way — it is to choose a variance that keeps the relay of activations numerically alive.

In [ ]:
x_w = np.random.normal(0, 1, size=(1000, 100))  # 1000 examples with 100 input features.
scales_w = [0.01, 0.1, 1.0]  # tiny, moderate, and huge raw standard deviations.
print("input variance:", round(float(np.var(x_w)), 3))  # should be close to 1.

▶ What you'll see: the input starts with variance near 1, giving us a clean reference scale.

In [ ]:
vars_w = []  # store output variances for each scale.
for s_w in scales_w:  # try each starting weight scale.
    W_w = np.random.normal(0, s_w, size=(100, 80))  # 100 inputs to 80 outputs.
    z_w = x_w @ W_w  # pre-activations before any nonlinearity.
    vars_w.append(float(np.var(z_w)))  # measure signal scale after one layer.
print("output variances:", [round(v, 3) for v in vars_w])  # tiny -> tiny, huge -> huge.
assert vars_w[0] < 0.02 and vars_w[2] > 50  # concrete failure modes.

▶ What you'll see: the output variance changes by orders of magnitude simply because the initial weight scale changed.

In [ ]:
plt.figure(figsize=(5, 3))  # compare scales visually.
plt.bar([str(s) for s in scales_w], vars_w, color=["crimson", "seagreen", "crimson"])  # variance by raw std.
plt.yscale("log")  # log scale makes shrinking and exploding visible together.
plt.xlabel("weight std")  # x-axis is the initializer scale.
plt.ylabel("output variance, log scale")  # y-axis is the signal variance after one layer.
plt.title("1: initialization scale controls signal variance")  # title the diagnostic.
plt.show()  # display the bar chart.

▶ What you'll see: too-small weights nearly erase the signal, while unit-scale weights blow it up.

*Why it's done this way:* for independent zero-mean inputs and weights, $\mathrm{Var}(z_j)=fan_{in}\,\mathrm{Var}(x)\,\mathrm{Var}(W)$. The factor $fan_{in}$ is the danger: summing many inputs multiplies variance unless the weight variance is divided by the number of incoming connections.

### 2. Xavier initialization preserves linear/tanh signal scale

Xavier initialization chooses $\mathrm{Var}(W)=2/(fan_{in}+fan_{out})$. For roughly balanced layers this is about $1/fan_{in}$, which cancels the variance growth from summing many inputs. It is designed for symmetric activations such as linear or tanh, where positive and negative values both pass information.

In [ ]:
fan_in_w, fan_out_w = 100, 80  # one dense layer shape.
xavier_var_w = 2 / (fan_in_w + fan_out_w)  # Xavier/Glorot variance.
xavier_std_w = np.sqrt(xavier_var_w)  # normal initializer standard deviation.
print("Xavier variance:", round(xavier_var_w, 5), "std:", round(xavier_std_w, 4))  # inspect formula.
assert round(xavier_var_w, 5) == 0.01111  # 2 / 180.

▶ What you'll see: Xavier's standard deviation is about 0.105, not an arbitrary small number.

In [ ]:
W_xav_w = np.random.normal(0, xavier_std_w, size=(fan_in_w, fan_out_w))  # Xavier normal weights.
z_xav_w = x_w @ W_xav_w  # one linear layer.
print("input variance:", round(float(np.var(x_w)), 3))  # reference.
print("Xavier output variance:", round(float(np.var(z_xav_w)), 3))  # should stay near 1.
assert 0.8 < np.var(z_xav_w) < 1.4  # stable one-layer scale.

▶ What you'll see: the output variance stays near the input variance instead of collapsing or exploding.

In [ ]:
plt.figure(figsize=(5, 3))  # histogram of pre-activation values.
plt.hist(z_xav_w.ravel(), bins=40, color="steelblue", alpha=0.8)  # distribution after Xavier layer.
plt.title("2: Xavier keeps pre-activations in a usable range")  # title the plot.
plt.xlabel("z = xW")  # label the activation axis.
plt.ylabel("count")  # label histogram counts.
plt.show()  # display the distribution.

▶ What you'll see: most pre-activations remain in a moderate range where tanh would not be instantly saturated.

*Why it's done this way:* Xavier balances forward and backward flow by considering both $fan_{in}$ and $fan_{out}$. Dividing by their average prevents the forward activations from growing with incoming width and prevents backward gradients from growing with outgoing width.

### 3. He initialization compensates for ReLU gating

ReLU keeps positive values and zeros negative values. If a zero-mean pre-activation is roughly symmetric, about half the units are switched off, so the second moment of the activation is roughly halved. He initialization doubles the Xavier-like incoming variance to $\mathrm{Var}(W)=2/fan_{in}$ so ReLU layers begin with a healthy post-activation scale.

In [ ]:
he_var_w = 2 / fan_in_w  # He/Kaiming variance for ReLU layers.
he_std_w = np.sqrt(he_var_w)  # normal initializer standard deviation.
print("He variance:", round(he_var_w, 4), "std:", round(he_std_w, 4))  # inspect formula.
assert round(he_var_w, 4) == 0.02  # 2 / 100.

▶ What you'll see: He uses a larger standard deviation than Xavier for the same incoming width.

In [ ]:
def relu_w(a_w):  # define ReLU for the walkthrough.
    return np.maximum(0, a_w)  # keep positive values and gate negatives to zero.

W_x_w = np.random.normal(0, xavier_std_w, size=(fan_in_w, fan_out_w))  # Xavier weights.
W_h_w = np.random.normal(0, he_std_w, size=(fan_in_w, fan_out_w))  # He weights.
a_x_w = relu_w(x_w @ W_x_w)  # ReLU after Xavier.
a_h_w = relu_w(x_w @ W_h_w)  # ReLU after He.
print("ReLU variance after Xavier:", round(float(np.var(a_x_w)), 3))  # lower scale.
print("ReLU variance after He:", round(float(np.var(a_h_w)), 3))  # healthier scale.
assert np.var(a_h_w) > np.var(a_x_w)  # He compensates for the gate.

▶ What you'll see: He initialization leaves larger ReLU activations because it anticipates that many values are zeroed.

In [ ]:
plt.figure(figsize=(5, 3))  # compare activation distributions.
plt.hist(a_x_w.ravel(), bins=40, alpha=0.65, label="Xavier + ReLU")  # lower-scale ReLU outputs.
plt.hist(a_h_w.ravel(), bins=40, alpha=0.65, label="He + ReLU")  # He-scaled ReLU outputs.
plt.title("3: He offsets ReLU's half-signal gate")  # title the histogram.
plt.xlabel("activation")  # label output activation values.
plt.ylabel("count")  # label counts.
plt.legend()  # identify the two initializers.
plt.show()  # display the comparison.

▶ What you'll see: the He distribution spreads farther right, keeping more usable signal after the ReLU gate.

*Why it's done this way:* ReLU is not scale-neutral: it discards negative responses and reduces the signal's second moment. The factor of 2 in $2/fan_{in}$ is the mathematical compensation for that expected loss.

### 4. Variance must survive many composed layers

A one-layer variance check is useful, but deep learning composes the same kind of map repeatedly. A small mismatch compounds: $0.8^{20}$ fades and $1.2^{20}$ explodes. The practical test is a forward pass through many randomly initialized layers.

In [ ]:
depth_w = 20  # number of layers in the toy deep stack.
width_w = 100  # keep every hidden layer the same width for clarity.
initializers_w = {"tiny": 0.03, "Xavier": np.sqrt(1 / width_w), "He": np.sqrt(2 / width_w), "large": 0.5}  # std choices.
print("initializers:", {k: round(v, 4) for k, v in initializers_w.items()})  # inspect raw scales.

▶ What you'll see: Xavier and He scales are tied to width; tiny and large are arbitrary baselines.

In [ ]:
curves_w = {}  # activation variance by layer for each initializer.
for name_w, std_w in initializers_w.items():  # run one deep stack per initializer.
    h_w = np.random.normal(0, 1, size=(512, width_w))  # fresh input batch.
    vals_w = [float(np.var(h_w))]  # start with input variance.
    for layer_w in range(depth_w):  # compose many layers.
        W_w = np.random.normal(0, std_w, size=(width_w, width_w))  # initialize the layer.
        h_w = relu_w(h_w @ W_w)  # ReLU network forward pass.
        vals_w.append(float(np.var(h_w)))  # record post-ReLU variance.
    curves_w[name_w] = vals_w  # save this initializer's curve.
print("final variances:", {k: round(v[-1], 6) for k, v in curves_w.items()})  # inspect depth effect.
assert curves_w["tiny"][-1] < 1e-8 and curves_w["large"][-1] > 1e6  # collapse/explosion.

▶ What you'll see: arbitrary scales fail dramatically after repeated composition.

In [ ]:
plt.figure(figsize=(5.5, 3.3))  # create a depth diagnostic plot.
for name_w, vals_w in curves_w.items():  # plot every initializer's variance path.
    plt.plot(vals_w, marker="o", markersize=2, label=name_w)  # one line per initializer.
plt.yscale("log")  # signal scale changes multiplicatively, so log scale is natural.
plt.xlabel("layer")  # label depth.
plt.ylabel("activation variance, log scale")  # label variance.
plt.title("4: small scale errors compound with depth")  # title the plot.
plt.legend()  # show initializer names.
plt.show()  # display the learning-signal relay.

▶ What you'll see: tiny collapses, large explodes, and variance-aware initializers are far more stable.

*Why it's done this way:* deep nets are products of many Jacobians. Initialization is chosen so the typical multiplier per layer is near 1; otherwise forward signals and backward gradients inherit an exponential depth problem before learning even starts.

### 5. Backward gradients have their own scale problem

Training needs gradients to travel backward through layers. If a layer's weights are too small, upstream gradients shrink; if too large, they explode. Initialization therefore protects both activations and gradients, not just pretty forward histograms.

In [ ]:
g_w = np.random.normal(0, 1, size=(512, width_w))  # gradient arriving from the output side.
back_scales_w = [0.03, np.sqrt(2 / width_w), 0.5]  # tiny, He-like, and large weight std.
print("incoming gradient variance:", round(float(np.var(g_w)), 3))  # reference gradient scale.

▶ What you'll see: the incoming gradient starts with variance near 1.

In [ ]:
back_vars_w = []  # store gradient variance after one backward matrix multiply.
for std_w in back_scales_w:  # test each backward scale.
    W_w = np.random.normal(0, std_w, size=(width_w, width_w))  # layer weights.
    relu_mask_w = (np.random.normal(0, 1, size=(512, width_w)) > 0).astype(float)  # approximate ReLU derivative mask.
    g_prev_w = (g_w * relu_mask_w) @ W_w.T  # backprop through ReLU and dense weights.
    back_vars_w.append(float(np.var(g_prev_w)))  # measure previous-layer gradient variance.
print("backward variances:", [round(v, 3) for v in back_vars_w])  # inspect shrink/explode behavior.
assert back_vars_w[0] < 0.1 and back_vars_w[2] > 5  # concrete gradient failure modes.

▶ What you'll see: bad weight scale damages gradients just as it damages activations.

In [ ]:
plt.figure(figsize=(5, 3))  # visualize backward variance by scale.
plt.bar(["tiny", "He-like", "large"], back_vars_w, color=["crimson", "seagreen", "crimson"])  # scale comparison.
plt.yscale("log")  # log scale shows both vanishing and exploding gradients.
plt.title("5: initialization also sets gradient scale")  # title the plot.
plt.ylabel("previous-gradient variance")  # label y-axis.
plt.show()  # display the comparison.

▶ What you'll see: the same variance logic governs whether gradients remain learnable.

*Why it's done this way:* the derivative of a dense layer contains $W^\top$, so the variance of $W$ directly controls the variance of backpropagated gradients. Xavier includes $fan_{out}$ and He includes the ReLU gate because the backward pass sees those same structural facts.

### 6. Orthogonal and LSUV: preserving directions, then calibrating data scale

Variance formulas assume independent random matrices and idealized input distributions. Orthogonal initialization starts from a matrix whose columns preserve lengths as well as possible, and LSUV (layer-sequential unit-variance) then rescales a layer using actual batch activations so its output variance is close to 1.

In [ ]:
A_w = np.random.normal(size=(80, 80))  # random square matrix.
Q_orth_w, _ = np.linalg.qr(A_w)  # QR decomposition gives an orthogonal Q.
identity_error_w = np.linalg.norm(Q_orth_w.T @ Q_orth_w - np.eye(80))  # check Q^T Q ≈ I.
print("orthogonality error:", round(float(identity_error_w), 10))  # nearly zero.
assert identity_error_w < 1e-10  # verify length-preserving columns.

▶ What you'll see: the orthogonal matrix passes a direct $Q^TQ=I$ check.

In [ ]:
v_w = np.random.normal(size=(200, 80))  # many vectors to test length preservation.
before_norm_w = np.linalg.norm(v_w, axis=1)  # lengths before multiplication.
after_norm_w = np.linalg.norm(v_w @ Q_orth_w, axis=1)  # lengths after orthogonal multiplication.
print("mean norm before:", round(float(before_norm_w.mean()), 3))  # reference.
print("mean norm after:", round(float(after_norm_w.mean()), 3))  # should match.
assert abs(before_norm_w.mean() - after_norm_w.mean()) < 1e-10  # orthogonal preserves norms.

▶ What you'll see: multiplying by an orthogonal matrix rotates/reflection-transforms vectors without changing their lengths.

In [ ]:
batch_w = np.random.normal(2.0, 3.0, size=(500, 80))  # non-ideal real batch scale.
out_w = batch_w @ Q_orth_w  # output before LSUV scaling.
lsuv_scale_w = 1 / np.sqrt(np.var(out_w) + 1e-8)  # one LSUV rescale factor.
out_l_w = out_w * lsuv_scale_w  # calibrated output.
print("variance before LSUV:", round(float(np.var(out_w)), 3))  # data-dependent scale.
print("variance after LSUV:", round(float(np.var(out_l_w)), 3))  # near 1.
assert round(float(np.var(out_l_w)), 3) == 1.0  # calibrated unit variance.

▶ What you'll see: LSUV turns the actual batch's layer output variance into approximately 1.

*Why it's done this way:* orthogonal initialization preserves geometric directions, while LSUV checks the data distribution the layer really sees. The formula-based initializer gives a principled start; the data-dependent rescale corrects mismatch between theory and the current batch.

## 🛠️ Setup

In [ ]:
import numpy as np # load NumPy for arrays, linear algebra, random initialization, and numerical assertions.
import matplotlib.pyplot as plt # load Matplotlib for histograms, bar charts, and signal-scale diagnostics.
np.random.seed(0) # make all stochastic examples reproducible across notebook runs.

## 🟢 Basics (warm-up)

### Basic 1 — Compute fan-in and fan-out

**Goal.** Read the shape of a dense weight matrix, because initialization formulas are shape-aware rather than fixed constants. We build it in 2 steps.

In [ ]:
shape_b1 = (4, 3) # represent a dense layer with 4 incoming features and 3 output units.
fan_in_b1, fan_out_b1 = shape_b1 # unpack the two dimensions used by initializer formulas.
print("fan_in:", fan_in_b1, "fan_out:", fan_out_b1) # inspect connection counts.

▶ What you'll see: the layer has 4 inputs feeding 3 outputs.

In [ ]:
xavier_var_b1 = 2 / (fan_in_b1 + fan_out_b1) # compute Xavier variance for this shape.
he_var_b1 = 2 / fan_in_b1 # compute He variance for this shape.
print("Xavier variance:", round(xavier_var_b1, 3), "He variance:", round(he_var_b1, 3)) # inspect both formulas.
assert round(xavier_var_b1, 3) == 0.286 and round(he_var_b1, 3) == 0.5 # self-check the arithmetic.
plt.figure(figsize=(4, 3)) # create a small formula comparison plot.
plt.bar(["Xavier", "He"], [xavier_var_b1, he_var_b1], color=["steelblue", "orange"]) # visualize variance choices.
plt.title("B1: fan counts set variance") # title the chart.
plt.ylabel("Var(W)") # label variance.
plt.show() # display the bar chart.

▶ What you'll see: He is larger because it prepares for ReLU's gating.

👀 Takeaway: initialization scale is computed from layer shape, especially fan-in and fan-out.

### Basic 2 — Draw small random weights

**Goal.** Sample zero-mean weights, because random signs break symmetry while the variance controls signal scale. We build it in 2 steps.

In [ ]:
std_b2 = 0.1 # choose a small standard deviation for a toy initializer.
W_b2 = np.random.normal(0, std_b2, size=(4, 3)) # draw weights independently from a normal distribution.
print("weights:\n", np.round(W_b2, 3)) # inspect the sampled matrix.

▶ What you'll see: weights have mixed signs and small magnitudes.

In [ ]:
print("sample mean:", round(float(W_b2.mean()), 3), "sample std:", round(float(W_b2.std()), 3)) # inspect empirical scale.
plt.figure(figsize=(4, 3)) # create a compact histogram.
plt.hist(W_b2.ravel(), bins=8, color="teal") # visualize the sampled weights.
plt.title("B2: random weights around zero") # title the plot.
plt.xlabel("weight value") # label the value axis.
plt.ylabel("count") # label counts.
plt.show() # display the histogram.

▶ What you'll see: the tiny sample is not exactly mean 0, but it is centered around 0.

👀 Takeaway: random initialization breaks unit symmetry; its variance decides how strongly units respond at the start.

### Basic 3 — Run one affine signal

**Goal.** Compute $xW+b$ by hand-sized arrays, because every initialization choice eventually changes these affine pre-activations. We build it in 2 steps.

In [ ]:
x_b3 = np.array([1.5, -0.5]) # use the lesson's two-input scratch vector.
w_b3 = np.array([1.2, 0.2]) # use visible weights for one output unit.
b_b3 = 0.8 # set the visible bias from the content block.
terms_b3 = x_b3 * w_b3 # compute per-input contributions.
print("terms:", terms_b3, "bias:", b_b3) # inspect the pieces before summing.

▶ What you'll see: the two weighted-input terms are 1.8 and -0.1.

In [ ]:
z_b3 = float(np.sum(terms_b3) + b_b3) # compute the affine pre-activation.
print("affine signal:", round(z_b3, 3)) # inspect the final scalar.
assert round(z_b3, 3) == 2.5 # self-check the lesson arithmetic.
plt.figure(figsize=(4, 3)) # create a contribution chart.
plt.bar(["1.2·1.5", "0.2·-0.5", "bias"], [terms_b3[0], terms_b3[1], b_b3], color="slateblue") # show additive parts.
plt.axhline(0, color="black", linewidth=0.8) # reference line for negative contribution.
plt.title("B3: affine signal pieces") # title the plot.
plt.xticks(rotation=20) # rotate labels for readability.
plt.show() # display the chart.

▶ What you'll see: the positive first input and the bias dominate the affine signal.

👀 Takeaway: initialization scale affects the weighted terms that feed every activation.

### Basic 4 — Apply a ReLU gate

**Goal.** Pass a pre-activation through ReLU, because He initialization exists specifically for gated rectified layers. We build it in 2 steps.

In [ ]:
z_b4 = np.array([-1.5, 0.0, 2.5, 4.0]) # define several pre-activation values.
a_b4 = np.maximum(0, z_b4) # apply ReLU elementwise.
print("z:", z_b4) # inspect raw signals.
print("ReLU(z):", a_b4) # inspect gated signals.

▶ What you'll see: negative values become zero while positive values pass through unchanged.

In [ ]:
active_fraction_b4 = float(np.mean(a_b4 > 0)) # measure how many units remain active.
print("active fraction:", round(active_fraction_b4, 2)) # inspect ReLU gate rate.
assert active_fraction_b4 == 0.5 # two of four values are positive.
plt.figure(figsize=(4, 3)) # create a before-after bar chart.
plt.bar(np.arange(len(z_b4)) - 0.15, z_b4, width=0.3, label="z", color="gray") # raw pre-activations.
plt.bar(np.arange(len(a_b4)) + 0.15, a_b4, width=0.3, label="ReLU(z)", color="seagreen") # gated activations.
plt.title("B4: ReLU gates negative signals") # title the plot.
plt.legend() # show labels.
plt.show() # display the comparison.

▶ What you'll see: half the values are shut off in this toy example.

👀 Takeaway: ReLU changes activation scale, so its initializer must account for the gate.

### Basic 5 — Compare raw scales in one layer

**Goal.** See one layer shrink or amplify input variance, because the same effect compounds with depth. We build it in 3 steps.

In [ ]:
X_b5 = np.random.normal(0, 1, size=(500, 50)) # create a batch with unit input variance.
stds_b5 = np.array([0.01, 0.1, 0.5]) # choose three candidate weight standard deviations.
print("input variance:", round(float(np.var(X_b5)), 3)) # inspect the reference variance.

▶ What you'll see: the input batch starts close to variance 1.

In [ ]:
out_vars_b5 = [] # collect output variance for each candidate std.
for std_b5 in stds_b5: # loop over scales.
    W_b5 = np.random.normal(0, std_b5, size=(50, 40)) # draw weights at this scale.
    Z_b5 = X_b5 @ W_b5 # compute pre-activations.
    out_vars_b5.append(float(np.var(Z_b5))) # store output variance.
print("output variances:", np.round(out_vars_b5, 3)) # inspect scale sensitivity.
assert out_vars_b5[0] < out_vars_b5[1] < out_vars_b5[2] # larger std gives larger output variance.

In [ ]:
plt.figure(figsize=(4, 3)) # create a compact scale plot.
plt.plot(stds_b5, out_vars_b5, marker="o", color="purple") # show variance by weight std.
plt.title("B5: output variance grows with weight scale") # title the plot.
plt.xlabel("weight std") # label x-axis.
plt.ylabel("Var(XW)") # label y-axis.
plt.show() # display the line chart.

▶ What you'll see: increasing weight standard deviation sharply raises output variance.

👀 Takeaway: initialization variance is a numerical control knob, not a cosmetic random choice.

### Basic 6 — Compute Xavier standard deviation

**Goal.** Convert Xavier variance into a standard deviation, because normal samplers use standard deviation as their scale parameter. We build it in 2 steps.

In [ ]:
fan_in_b6, fan_out_b6 = 64, 32 # define a layer shape.
var_b6 = 2 / (fan_in_b6 + fan_out_b6) # compute Xavier variance.
std_b6 = np.sqrt(var_b6) # convert variance to standard deviation.
print("variance:", round(var_b6, 5), "std:", round(std_b6, 5)) # inspect both values.
assert round(var_b6, 5) == 0.02083 # self-check 2/96.

▶ What you'll see: the standard deviation is the square root of the formula's variance.

In [ ]:
W_b6 = np.random.normal(0, std_b6, size=(fan_in_b6, fan_out_b6)) # sample Xavier normal weights.
print("sample std:", round(float(W_b6.std()), 4)) # inspect empirical scale.
plt.figure(figsize=(4, 3)) # create a weight histogram.
plt.hist(W_b6.ravel(), bins=30, color="steelblue") # visualize sampled values.
plt.title("B6: Xavier normal weights") # title the plot.
plt.xlabel("weight") # label values.
plt.show() # display the histogram.

▶ What you'll see: sampled weights cluster around 0 with spread near the Xavier standard deviation.

👀 Takeaway: Xavier initialization is a variance formula plus a random distribution with that variance.

### Basic 7 — Compute He standard deviation

**Goal.** Compute the ReLU-specific scale $\sqrt{2/fan_{in}}$, because rectified units need a larger start than linear units. We build it in 2 steps.

In [ ]:
fan_in_b7 = 64 # define incoming connections.
var_b7 = 2 / fan_in_b7 # compute He variance.
std_b7 = np.sqrt(var_b7) # convert to standard deviation.
print("He variance:", round(var_b7, 5), "std:", round(std_b7, 5)) # inspect the initializer.
assert round(var_b7, 5) == 0.03125 # self-check 2/64.

▶ What you'll see: He standard deviation is larger than Xavier's for a 64-to-32 layer.

In [ ]:
W_b7 = np.random.normal(0, std_b7, size=(fan_in_b7, 32)) # sample He normal weights.
X_b7 = np.random.normal(0, 1, size=(400, fan_in_b7)) # create unit-variance inputs.
A_b7 = np.maximum(0, X_b7 @ W_b7) # compute ReLU activations.
print("ReLU activation variance:", round(float(np.var(A_b7)), 3)) # inspect post-gate scale.
plt.figure(figsize=(4, 3)) # create a histogram of activations.
plt.hist(A_b7.ravel(), bins=35, color="orange") # visualize ReLU outputs.
plt.title("B7: He + ReLU activations") # title the plot.
plt.show() # display the histogram.

▶ What you'll see: many activations are zero, but the positive side remains spread out.

👀 Takeaway: He initialization bakes ReLU's expected signal loss into the starting variance.

### Basic 8 — Show sigmoid saturation from large weights

**Goal.** Observe saturation, because overly large pre-activations make sigmoid-like units nearly flat and gradients tiny. We build it in 3 steps.

In [ ]:
z_b8 = np.linspace(-8, 8, 200) # sweep pre-activation values from very negative to very positive.
sig_b8 = 1 / (1 + np.exp(-z_b8)) # compute sigmoid values using NumPy.
deriv_b8 = sig_b8 * (1 - sig_b8) # derivative of sigmoid.
print("max derivative:", round(float(deriv_b8.max()), 3)) # inspect best-case gradient.
assert round(float(deriv_b8.max()), 3) == 0.25 # sigmoid derivative peaks at 1/4.

▶ What you'll see: even the best sigmoid derivative is only 0.25.

In [ ]:
sat_mask_b8 = np.abs(z_b8) > 5 # mark extreme pre-activations.
print("mean derivative when |z|>5:", round(float(np.mean(deriv_b8[sat_mask_b8])), 4)) # inspect saturated slope.
assert np.mean(deriv_b8[sat_mask_b8]) < 0.01 # saturated units have tiny gradients.

In [ ]:
plt.figure(figsize=(5, 3)) # create a sigmoid diagnostic plot.
plt.plot(z_b8, sig_b8, label="sigmoid", color="teal") # plot activation.
plt.plot(z_b8, deriv_b8, label="derivative", color="crimson") # plot slope.
plt.title("B8: large |z| saturates sigmoid") # title the plot.
plt.xlabel("pre-activation z") # label x-axis.
plt.legend() # show curve labels.
plt.show() # display the plot.

▶ What you'll see: the sigmoid flattens near 0 and 1, and the derivative nearly vanishes there.

👀 Takeaway: initialization should keep early pre-activations out of saturated regions.

### Basic 9 — Make one gradient-descent nudge

**Goal.** Update one scalar parameter, because initialization only starts training; learning then moves parameters by repeated small gradient steps. We build it in 2 steps.

In [ ]:
w_b9 = 2.0 # starting scalar parameter from the content block.
eta_b9 = 0.06 # learning rate from the content block.
g_b9 = 1.8 # gradient from the content block.
print("before:", w_b9, "eta:", eta_b9, "gradient:", g_b9) # inspect update ingredients.

▶ What you'll see: the update uses a small learning-rate multiplier.

In [ ]:
w_new_b9 = w_b9 - eta_b9 * g_b9 # gradient descent update.
print("after:", round(w_new_b9, 3)) # inspect the moved parameter.
assert round(w_new_b9, 3) == 1.892 # self-check the content-block arithmetic.
plt.figure(figsize=(4, 3)) # create a before-after plot.
plt.bar(["before", "after"], [w_b9, w_new_b9], color=["gray", "seagreen"]) # visualize the nudge.
plt.title("B9: one reliable nudge") # title the plot.
plt.ylabel("parameter value") # label the parameter scale.
plt.show() # display the comparison.

▶ What you'll see: the parameter moves by 0.108, not by the full gradient.

👀 Takeaway: stable initialization and stable learning rates both aim for repeated reliable nudges.

### Basic 10 — Normalize one activation

**Goal.** Standardize a signal with a mean and variance, because initialization and normalization both manage numerical scale. We build it in 2 steps.

In [ ]:
value_b10 = 2.5 # lesson signal from the scratch pass.
mean_b10 = 1.0 # reference mean.
var_b10 = 0.25 # reference variance.
eps_b10 = 1e-5 # small constant for numerical safety.
normalized_b10 = (value_b10 - mean_b10) / np.sqrt(var_b10 + eps_b10) # normalize the scalar.
print("normalized value:", round(float(normalized_b10), 3)) # inspect standardized signal.
assert round(float(normalized_b10), 3) == 3.0 # self-check the content-block value.

▶ What you'll see: the signal is three standard deviations above the reference mean.

In [ ]:
plt.figure(figsize=(4, 3)) # create a compact normalization plot.
plt.bar(["raw", "mean", "normalized"], [value_b10, mean_b10, normalized_b10], color=["orange", "gray", "teal"]) # compare quantities.
plt.title("B10: scale bookkeeping") # title the plot.
plt.ylabel("value") # label values.
plt.show() # display the bar chart.

▶ What you'll see: normalization changes the interpretation from raw magnitude to relative scale.

👀 Takeaway: scale management appears in initialization, normalization, and gradient flow.

## 🟡 Easy

### Easy 1 — Simulate Xavier through a linear stack

**Goal.** Track activation variance across several linear layers, because Xavier is designed to keep that variance roughly stable without ReLU gating. We build it in 4 steps.

In [ ]:
X_e1 = np.random.normal(0, 1, size=(600, 64)) # create a unit-variance input batch.
width_e1 = 64 # use square layers so fan_in equals fan_out.
depth_e1 = 8 # choose a modest stack depth.
print("start variance:", round(float(np.var(X_e1)), 3)) # inspect input scale.

▶ What you'll see: the input variance is close to 1.

In [ ]:
h_e1 = X_e1.copy() # start the forward pass.
vars_e1 = [float(np.var(h_e1))] # record input variance.
for layer_e1 in range(depth_e1): # pass through linear layers.
    std_e1 = np.sqrt(2 / (width_e1 + width_e1)) # Xavier std for a square layer.
    W_e1 = np.random.normal(0, std_e1, size=(width_e1, width_e1)) # sample Xavier weights.
    h_e1 = h_e1 @ W_e1 # linear forward pass.
    vars_e1.append(float(np.var(h_e1))) # record variance after this layer.
print("variance curve:", np.round(vars_e1, 3)) # inspect scale over depth.

In [ ]:
ratio_e1 = vars_e1[-1] / vars_e1[0] # summarize final scale drift.
print("final/start ratio:", round(float(ratio_e1), 3)) # inspect drift.
assert 0.2 < ratio_e1 < 5.0 # Xavier should avoid catastrophic collapse/explosion in this small demo.

In [ ]:
plt.figure(figsize=(5, 3)) # create the variance curve.
plt.plot(vars_e1, marker="o", color="steelblue") # plot variance by layer.
plt.axhline(1, color="gray", linestyle="--") # reference unit variance.
plt.title("E1: Xavier through linear layers") # title the plot.
plt.xlabel("layer") # label depth.
plt.ylabel("activation variance") # label variance.
plt.show() # display the curve.

▶ What you'll see: variance wanders but stays far from total collapse or explosion.

👀 Takeaway: Xavier is a linear/tanh-friendly default because it counters fan-in variance growth.

### Easy 2 — Simulate He through a ReLU stack

**Goal.** Track ReLU activation scale with He initialization, because the formula compensates for half the signal being gated off. We build it in 4 steps.

In [ ]:
X_e2 = np.random.normal(0, 1, size=(600, 64)) # create a unit-variance batch.
width_e2 = 64 # set hidden width.
depth_e2 = 8 # set number of ReLU layers.
print("start second moment:", round(float(np.mean(X_e2 ** 2)), 3)) # inspect raw energy.

▶ What you'll see: the input second moment is close to 1.

In [ ]:
h_e2 = X_e2.copy() # initialize activations.
second_e2 = [float(np.mean(h_e2 ** 2))] # store second moment, more natural for ReLU.
active_e2 = [] # store active fractions.
for layer_e2 in range(depth_e2): # compose ReLU layers.
    W_e2 = np.random.normal(0, np.sqrt(2 / width_e2), size=(width_e2, width_e2)) # He weights.
    h_e2 = np.maximum(0, h_e2 @ W_e2) # dense plus ReLU.
    second_e2.append(float(np.mean(h_e2 ** 2))) # record activation energy.
    active_e2.append(float(np.mean(h_e2 > 0))) # record ReLU activity.
print("second moments:", np.round(second_e2, 3)) # inspect scale preservation.
print("active fractions:", np.round(active_e2, 3)) # inspect gate rates.

In [ ]:
assert 0.2 < second_e2[-1] < 5.0 # He should avoid catastrophic scale failure in this toy stack.
print("final second moment:", round(second_e2[-1], 3)) # summarize final activation energy.

In [ ]:
plt.figure(figsize=(5, 3)) # create a depth plot.
plt.plot(second_e2, marker="o", color="orange") # plot second moment by layer.
plt.axhline(1, color="gray", linestyle="--") # reference input energy.
plt.title("E2: He through ReLU layers") # title the plot.
plt.xlabel("layer") # label layer index.
plt.ylabel("mean activation²") # label second moment.
plt.show() # display the curve.

▶ What you'll see: activation energy stays in a usable range across the ReLU stack.

👀 Takeaway: He initialization is matched to ReLU's gating, so it is usually preferred for rectified networks.

### Easy 3 — Compare Xavier and He after ReLU

**Goal.** Put Xavier and He side by side under ReLU, because an initializer should match the activation function rather than the layer in isolation. We build it in 3 steps.

In [ ]:
X_e3 = np.random.normal(0, 1, size=(800, 100)) # create a batch of unit-variance inputs.
fan_e3 = 100 # set square layer width.
std_x_e3 = np.sqrt(1 / fan_e3) # Xavier std for a square layer.
std_h_e3 = np.sqrt(2 / fan_e3) # He std for ReLU.
print("Xavier std:", round(std_x_e3, 3), "He std:", round(std_h_e3, 3)) # inspect scales.

▶ What you'll see: He's standard deviation is sqrt(2) times Xavier's in a square layer.

In [ ]:
A_x_e3 = np.maximum(0, X_e3 @ np.random.normal(0, std_x_e3, size=(fan_e3, fan_e3))) # ReLU after Xavier.
A_h_e3 = np.maximum(0, X_e3 @ np.random.normal(0, std_h_e3, size=(fan_e3, fan_e3))) # ReLU after He.
vars_e3 = np.array([np.var(A_x_e3), np.var(A_h_e3)]) # compare activation variances.
print("ReLU variances [Xavier, He]:", np.round(vars_e3, 3)) # inspect result.
assert vars_e3[1] > vars_e3[0] # He should produce the larger post-ReLU variance.

In [ ]:
plt.figure(figsize=(4, 3)) # create a comparison plot.
plt.bar(["Xavier+ReLU", "He+ReLU"], vars_e3, color=["steelblue", "orange"]) # visualize activation variance.
plt.title("E3: activation-aware initializer") # title the plot.
plt.ylabel("activation variance") # label scale.
plt.xticks(rotation=15) # rotate labels.
plt.show() # display the bars.

▶ What you'll see: Xavier produces smaller ReLU activation variance than He.

👀 Takeaway: activation functions change signal statistics, so initialization should be activation-aware.

### Easy 4 — Build an orthogonal initializer

**Goal.** Construct an orthogonal matrix with NumPy, because orthogonal initialization preserves vector lengths and avoids correlated directions. We build it in 3 steps.

In [ ]:
A_e4 = np.random.normal(size=(32, 32)) # draw a random square matrix.
Q_e4, R_e4 = np.linalg.qr(A_e4) # factor it into an orthogonal matrix and triangular matrix.
print("Q shape:", Q_e4.shape) # inspect initializer shape.

▶ What you'll see: QR returns a square matrix that can serve as an orthogonal weight matrix.

In [ ]:
err_e4 = float(np.linalg.norm(Q_e4.T @ Q_e4 - np.eye(32))) # compute orthogonality error.
print("orthogonality error:", round(err_e4, 12)) # inspect numerical precision.
assert err_e4 < 1e-10 # verify Q^TQ is identity up to floating-point precision.

In [ ]:
v_e4 = np.random.normal(size=(100, 32)) # create test vectors.
ratio_e4 = np.linalg.norm(v_e4 @ Q_e4, axis=1) / np.linalg.norm(v_e4, axis=1) # length after/before.
print("mean norm ratio:", round(float(ratio_e4.mean()), 6)) # inspect length preservation.
plt.figure(figsize=(4, 3)) # create norm-ratio histogram.
plt.hist(ratio_e4, bins=20, color="teal") # visualize ratios.
plt.title("E4: orthogonal length preservation") # title the plot.
plt.xlabel("||vQ|| / ||v||") # label ratio.
plt.show() # display the histogram.

▶ What you'll see: norm ratios cluster exactly around 1.

👀 Takeaway: orthogonal initialization preserves geometry instead of merely matching scalar variance.

### Easy 5 — Apply one LSUV rescale

**Goal.** Rescale a layer from measured batch variance, because LSUV uses actual data to correct the output scale. We build it in 4 steps.

In [ ]:
X_e5 = np.random.normal(3.0, 2.0, size=(500, 40)) # create a batch that is not unit-normal ideal.
W_e5 = np.random.normal(0, np.sqrt(2 / 40), size=(40, 40)) # start from He-like weights.
Y_e5 = X_e5 @ W_e5 # compute layer output before LSUV.
print("pre-LSUV variance:", round(float(np.var(Y_e5)), 3)) # inspect actual batch scale.

▶ What you'll see: the output variance may not be exactly 1 because the input batch is not idealized.

In [ ]:
scale_e5 = 1 / np.sqrt(np.var(Y_e5) + 1e-8) # compute LSUV scale factor.
W_l_e5 = W_e5 * scale_e5 # rescale the layer weights.
Y_l_e5 = X_e5 @ W_l_e5 # recompute outputs after rescaling.
print("LSUV scale:", round(float(scale_e5), 3)) # inspect the multiplicative correction.
print("post-LSUV variance:", round(float(np.var(Y_l_e5)), 3)) # verify calibration.
assert round(float(np.var(Y_l_e5)), 3) == 1.0 # one LSUV pass gives unit variance here.

In [ ]:
means_e5 = [float(np.mean(Y_e5)), float(np.mean(Y_l_e5))] # compare means before/after.
vars_e5 = [float(np.var(Y_e5)), float(np.var(Y_l_e5))] # compare variances before/after.
print("means:", np.round(means_e5, 3), "variances:", np.round(vars_e5, 3)) # inspect both summaries.

In [ ]:
plt.figure(figsize=(4, 3)) # create a variance comparison chart.
plt.bar(["before", "after"], vars_e5, color=["crimson", "seagreen"]) # show calibration effect.
plt.axhline(1, color="black", linestyle="--") # reference target variance.
plt.title("E5: LSUV calibrates output variance") # title the plot.
plt.ylabel("batch output variance") # label variance.
plt.show() # display the chart.

▶ What you'll see: after one rescale, the batch output variance lands at 1.

👀 Takeaway: LSUV combines principled initialization with data-dependent scale calibration.

## 🔴 Advanced

### Advanced 1 — Sweep depth for vanishing and exploding signals

**Goal.** Quantify depth sensitivity across several raw scales, because a small per-layer mismatch becomes exponential in deep networks. We build it in 4 steps.

In [ ]:
width_a1 = 80 # set hidden width.
depth_a1 = 25 # set stack depth.
scales_a1 = np.array([0.02, np.sqrt(2 / width_a1), 0.35]) # tiny, He, and too-large scales.
labels_a1 = ["tiny", "He", "large"] # names for plotting.
print("scales:", np.round(scales_a1, 4)) # inspect standard deviations.

▶ What you'll see: the He scale is width-dependent and sits between arbitrary extremes.

In [ ]:
curves_a1 = [] # store one variance curve per scale.
for std_a1 in scales_a1: # evaluate every initializer scale.
    h_a1 = np.random.normal(0, 1, size=(400, width_a1)) # fresh input batch.
    curve_a1 = [float(np.var(h_a1))] # start with input variance.
    for layer_a1 in range(depth_a1): # compose many ReLU layers.
        W_a1 = np.random.normal(0, std_a1, size=(width_a1, width_a1)) # sample current layer.
        h_a1 = np.maximum(0, h_a1 @ W_a1) # ReLU forward pass.
        curve_a1.append(float(np.var(h_a1))) # store activation variance.
    curves_a1.append(curve_a1) # save this scale's curve.
print("final variances:", [round(c[-1], 6) for c in curves_a1]) # inspect depth outcome.

In [ ]:
assert curves_a1[0][-1] < 1e-10 # tiny scale vanishes.
assert curves_a1[2][-1] > 1e5 # large scale explodes.
print("He final variance:", round(curves_a1[1][-1], 3)) # inspect the variance-aware scale.

In [ ]:
plt.figure(figsize=(5.5, 3.2)) # create the depth sweep plot.
for label_a1, curve_a1 in zip(labels_a1, curves_a1): # plot each scale.
    plt.plot(curve_a1, label=label_a1) # draw variance by layer.
plt.yscale("log") # exponential effects are easiest on log scale.
plt.title("A1: depth amplifies scale mistakes") # title the plot.
plt.xlabel("layer") # label depth.
plt.ylabel("activation variance") # label variance.
plt.legend() # show scale labels.
plt.show() # display the chart.

▶ What you'll see: tiny and large scales fail by depth, while the variance-aware choice stays far more controlled.

👀 Takeaway: initialization is a global training-stability decision made through local layer variance.

### Advanced 2 — Compare forward and backward variance together

**Goal.** Measure both activation and gradient scale, because an initializer that helps only one direction can still train poorly. We build it in 5 steps.

In [ ]:
width_a2 = 100 # set a square layer width.
X_a2 = np.random.normal(0, 1, size=(700, width_a2)) # input activations.
G_a2 = np.random.normal(0, 1, size=(700, width_a2)) # output-side gradients.
stds_a2 = np.array([np.sqrt(1 / width_a2), np.sqrt(2 / width_a2)]) # Xavier-like and He-like for square layers.
print("stds [Xavier, He]:", np.round(stds_a2, 4)) # inspect compared initializers.

▶ What you'll see: He is larger by a factor of sqrt(2).

In [ ]:
forward_vars_a2 = [] # store post-ReLU activation variances.
backward_vars_a2 = [] # store previous-gradient variances.
for std_a2 in stds_a2: # compare the two scales.
    W_a2 = np.random.normal(0, std_a2, size=(width_a2, width_a2)) # sample weights.
    Z_a2 = X_a2 @ W_a2 # pre-activation.
    A_a2 = np.maximum(0, Z_a2) # ReLU activation.
    M_a2 = (Z_a2 > 0).astype(float) # ReLU derivative mask.
    G_prev_a2 = (G_a2 * M_a2) @ W_a2.T # backpropagate through ReLU and dense weights.
    forward_vars_a2.append(float(np.var(A_a2))) # record forward scale.
    backward_vars_a2.append(float(np.var(G_prev_a2))) # record backward scale.
print("forward variances:", np.round(forward_vars_a2, 3)) # inspect activation scale.
print("backward variances:", np.round(backward_vars_a2, 3)) # inspect gradient scale.

In [ ]:
assert forward_vars_a2[1] > forward_vars_a2[0] # He gives larger ReLU activations.
assert backward_vars_a2[1] > backward_vars_a2[0] # He gives larger backpropagated gradients.
print("forward/backward ratios:", round(forward_vars_a2[1] / forward_vars_a2[0], 2), round(backward_vars_a2[1] / backward_vars_a2[0], 2)) # summarize effect.

In [ ]:
x_a2 = np.arange(2) # positions for grouped bars.
plt.figure(figsize=(5, 3)) # create grouped comparison.
plt.bar(x_a2 - 0.18, forward_vars_a2, width=0.36, label="forward", color="orange") # activation variance bars.
plt.bar(x_a2 + 0.18, backward_vars_a2, width=0.36, label="backward", color="purple") # gradient variance bars.
plt.xticks(x_a2, ["Xavier", "He"]) # label initializers.
plt.title("A2: forward and backward scale") # title the plot.
plt.ylabel("variance") # label y-axis.
plt.legend() # show bar labels.
plt.show() # display the grouped bars.

▶ What you'll see: changing initializer scale affects both activations and backpropagated gradients.

👀 Takeaway: initialization is chosen for the whole optimization pipeline, not just the first forward pass.

### Advanced 3 — Inspect Jacobian singular values

**Goal.** Compare random Gaussian and orthogonal matrices by singular values, because gradient norms are controlled by products of layer Jacobians. We build it in 4 steps.

In [ ]:
n_a3 = 60 # square matrix size.
G_a3 = np.random.normal(0, np.sqrt(1 / n_a3), size=(n_a3, n_a3)) # Gaussian Xavier-like matrix.
Q_a3, _ = np.linalg.qr(np.random.normal(size=(n_a3, n_a3))) # orthogonal matrix.
print("matrix size:", n_a3) # inspect the Jacobian dimension.

▶ What you'll see: both candidate matrices map 60-dimensional vectors to 60-dimensional vectors.

In [ ]:
sv_g_a3 = np.linalg.svd(G_a3, compute_uv=False) # singular values for Gaussian matrix.
sv_q_a3 = np.linalg.svd(Q_a3, compute_uv=False) # singular values for orthogonal matrix.
print("Gaussian sv min/max:", round(float(sv_g_a3.min()), 3), round(float(sv_g_a3.max()), 3)) # inspect spread.
print("Orthogonal sv min/max:", round(float(sv_q_a3.min()), 3), round(float(sv_q_a3.max()), 3)) # inspect preservation.

In [ ]:
assert abs(float(sv_q_a3.min()) - 1) < 1e-10 and abs(float(sv_q_a3.max()) - 1) < 1e-10 # all orthogonal singular values are 1.
spread_a3 = float(sv_g_a3.max() / sv_g_a3.min()) # quantify Gaussian anisotropy.
print("Gaussian condition ratio:", round(spread_a3, 2)) # inspect stretching unevenness.

In [ ]:
plt.figure(figsize=(5, 3)) # create singular-value histogram.
plt.hist(sv_g_a3, bins=20, alpha=0.7, label="Gaussian", color="crimson") # random Gaussian singular values.
plt.hist(sv_q_a3, bins=20, alpha=0.7, label="Orthogonal", color="teal") # orthogonal singular values.
plt.title("A3: singular values of initial Jacobians") # title the plot.
plt.xlabel("singular value") # label singular-value axis.
plt.legend() # show labels.
plt.show() # display the histogram.

▶ What you'll see: orthogonal singular values sit at 1, while Gaussian singular values spread across contractions and expansions.

👀 Takeaway: orthogonal initialization can make the initial layer Jacobian more length-preserving than an unconstrained Gaussian draw.

### Advanced 4 — Calibrate a two-layer network with LSUV

**Goal.** Apply sequential unit-variance rescaling to two layers, because LSUV calibrates each layer using the activations produced by earlier calibrated layers. We build it in 5 steps.

In [ ]:
X_a4 = np.random.normal(1.5, 2.5, size=(600, 50)) # create non-standardized input data.
W1_a4 = np.random.normal(0, np.sqrt(2 / 50), size=(50, 40)) # first He-style layer.
W2_a4 = np.random.normal(0, np.sqrt(2 / 40), size=(40, 30)) # second He-style layer.
print("input variance:", round(float(np.var(X_a4)), 3)) # inspect data scale.

▶ What you'll see: the actual input variance is much larger than 1.

In [ ]:
H1_a4 = np.maximum(0, X_a4 @ W1_a4) # first hidden activations before LSUV.
scale1_a4 = 1 / np.sqrt(np.var(H1_a4) + 1e-8) # LSUV scale for layer 1 output.
W1_a4 = W1_a4 * scale1_a4 # rescale first layer weights.
H1_l_a4 = np.maximum(0, X_a4 @ W1_a4) # recompute first hidden activations.
print("layer1 variance before/after:", round(float(np.var(H1_a4)), 3), round(float(np.var(H1_l_a4)), 3)) # inspect calibration.

In [ ]:
H2_a4 = np.maximum(0, H1_l_a4 @ W2_a4) # second activations using calibrated first layer.
scale2_a4 = 1 / np.sqrt(np.var(H2_a4) + 1e-8) # LSUV scale for second layer output.
W2_a4 = W2_a4 * scale2_a4 # rescale second layer weights.
H2_l_a4 = np.maximum(0, H1_l_a4 @ W2_a4) # recompute second activations.
print("layer2 variance before/after:", round(float(np.var(H2_a4)), 3), round(float(np.var(H2_l_a4)), 3)) # inspect calibration.

In [ ]:
assert abs(np.var(H1_l_a4) - 1) < 1e-6 # first layer calibrated.
assert abs(np.var(H2_l_a4) - 1) < 1e-6 # second layer calibrated.
vars_before_after_a4 = [float(np.var(H1_a4)), float(np.var(H1_l_a4)), float(np.var(H2_a4)), float(np.var(H2_l_a4))] # collect plot values.
print("variance summary:", np.round(vars_before_after_a4, 3)) # inspect all values.

In [ ]:
plt.figure(figsize=(5, 3)) # create a calibration plot.
plt.bar(["L1 before", "L1 after", "L2 before", "L2 after"], vars_before_after_a4, color=["crimson", "seagreen", "crimson", "seagreen"]) # compare before/after.
plt.axhline(1, color="black", linestyle="--") # target variance.
plt.title("A4: sequential LSUV calibration") # title the plot.
plt.ylabel("activation variance") # label y-axis.
plt.xticks(rotation=20) # rotate labels.
plt.show() # display the bars.

▶ What you'll see: each layer is rescaled to unit variance using the activations it actually receives.

👀 Takeaway: LSUV is layer-sequential because downstream layers should be calibrated after upstream scales are fixed.

### Advanced 5 — Estimate activation memory cost

**Goal.** Compute activation memory from batch, layer count, width, and dtype size, because deep-learning scale choices eventually become hardware constraints. We build it in 4 steps.

In [ ]:
vectors_a5 = 5 # number of activation vectors from the content block.
width_a5 = 128 # vector length from the content block.
bytes_per_float_a5 = 4 # 32-bit float storage.
kb_a5 = vectors_a5 * width_a5 * bytes_per_float_a5 / 1024 # compute memory in KB.
print("memory KB:", round(kb_a5, 3)) # inspect small example.
assert round(kb_a5, 3) == 2.5 # self-check the content-block arithmetic.

▶ What you'll see: five length-128 float32 vectors take 2.5 KB.

In [ ]:
batch_sizes_a5 = np.array([32, 128, 512]) # compare larger batches.
layers_a5 = 24 # choose a deeper network.
width_big_a5 = 1024 # choose a more realistic hidden width.
mb_a5 = batch_sizes_a5 * layers_a5 * width_big_a5 * bytes_per_float_a5 / (1024 ** 2) # activation memory in MB.
print("activation MB:", np.round(mb_a5, 1)) # inspect scaling with batch size.

In [ ]:
assert np.all(np.diff(mb_a5) > 0) # memory rises with batch size.
ratio_a5 = mb_a5[-1] / mb_a5[0] # compare largest to smallest batch.
print("largest/smallest memory ratio:", round(float(ratio_a5), 1)) # inspect linear scaling.

In [ ]:
plt.figure(figsize=(5, 3)) # create a hardware-scale plot.
plt.bar([str(b) for b in batch_sizes_a5], mb_a5, color="darkorange") # show memory by batch size.
plt.title("A5: activation memory scales linearly") # title the plot.
plt.xlabel("batch size") # label batch axis.
plt.ylabel("activation memory (MB)") # label memory axis.
plt.show() # display the chart.

▶ What you'll see: activation memory grows linearly with batch size, depth, and width.

👀 Takeaway: good initialization keeps numbers trainable, but model shape still has to fit memory budgets.

---

# Reference walkthrough — original compact notebook

The sections above build every idea from scratch with detailed steps and worked examples. Below is the original compact notebook for this lesson, kept as a concise reference and for its practice prompts.

Initialization chooses variance so signals and gradients begin neither fading nor exploding.

A fixed architecture can train or fail before the first update depending on scale. Xavier, He, orthogonal, and LSUV are ways to preserve useful signal magnitude. Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits
from sklearn.datasets import make_blobs
from sklearn.datasets import make_moons
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(6)


def clf_digits_ladder():
    rungs = []

    x1 = np.array([[0.0, 0.0], [1.0, 1.0], [0.0, 1.0], [1.0, 0.0]])
    y1 = np.array([0, 0, 1, 1])
    rungs.append(("D1 XOR", x1, y1))

    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=1.0, random_state=1)
    rungs.append(("D2 blobs (3-class)", x2, y2))

    x3, y3 = make_moons(n_samples=300, noise=0.3, random_state=2)
    rungs.append(("D3 noisy moons", x3, y3))

    digits = load_digits()
    xd = digits.data / 16.0
    rungs.append(("D4 digits (real, 10-class, 64-D)", xd, digits.target))

    rng = np.random.default_rng(5)
    xn = xd + rng.normal(0.0, 0.25, size=xd.shape)
    yn = digits.target.copy()
    flip = rng.random(yn.shape) < 0.1
    yn[flip] = rng.integers(0, 10, size=int(flip.sum()))
    rungs.append(("D5 digits + label/feature noise", xn, yn))

    return rungs


def clf_accuracy(build_and_predict, X, y):
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    preds = build_and_predict(x_tr, y_tr, x_te)
    return accuracy_score(y_te, preds)


def split_scale(X, y):
    if len(y) > 300:
        x_small, _, y_small, _ = train_test_split(X, y, train_size=300, random_state=6, stratify=y)
    else:
        x_small = X
        y_small = y
    x_tr, x_te, y_tr, y_te = train_test_split(x_small, y_small, test_size=0.4, random_state=0, stratify=y_small)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    return x_tr, x_te, y_tr, y_te


def one_hot(y, classes):
    out = np.zeros((len(y), classes))
    out[np.arange(len(y)), y] = 1.0
    return out


def softmax(z):
    z = z - np.max(z, axis=1, keepdims=True)
    ez = np.exp(z)
    return ez / np.sum(ez, axis=1, keepdims=True)


def relu(z):
    return np.maximum(z, 0.0)


def init_weights(n_in, n_hidden, n_out, mode, seed):
    rng = np.random.default_rng(seed)
    if mode == "xavier":
        scale1 = math.sqrt(2.0 / (n_in + n_hidden))
        scale2 = math.sqrt(2.0 / (n_hidden + n_out))
        W1 = rng.normal(0.0, scale1, size=(n_in, n_hidden))
        W2 = rng.normal(0.0, scale2, size=(n_hidden, n_out))
    elif mode == "he":
        scale1 = math.sqrt(2.0 / n_in)
        scale2 = math.sqrt(2.0 / n_hidden)
        W1 = rng.normal(0.0, scale1, size=(n_in, n_hidden))
        W2 = rng.normal(0.0, scale2, size=(n_hidden, n_out))
    elif mode == "tiny":
        W1 = rng.normal(0.0, 0.01, size=(n_in, n_hidden))
        W2 = rng.normal(0.0, 0.01, size=(n_hidden, n_out))
    elif mode == "large":
        W1 = rng.normal(0.0, 2.0, size=(n_in, n_hidden))
        W2 = rng.normal(0.0, 2.0, size=(n_hidden, n_out))
    elif mode == "orthogonal":
        Q1, _ = np.linalg.qr(rng.normal(size=(n_in, max(n_in, n_hidden))))
        Q2, _ = np.linalg.qr(rng.normal(size=(n_hidden, max(n_hidden, n_out))))
        W1 = Q1[:, :n_hidden]
        W2 = Q2[:, :n_out]
    else:
        W1 = rng.normal(0.0, 0.1, size=(n_in, n_hidden))
        W2 = rng.normal(0.0, 0.1, size=(n_hidden, n_out))
    b1 = np.zeros(n_hidden)
    b2 = np.zeros(n_out)
    return {"W1": W1, "b1": b1, "W2": W2, "b2": b2}


def forward(params, X, dropout_p=0.0, rng=None, dropconnect_p=0.0):
    W1 = params["W1"]
    if dropconnect_p > 0.0 and rng is not None:
        keep_w = 1.0 - dropconnect_p
        mask_w = rng.binomial(1, keep_w, size=W1.shape) / keep_w
        W1 = W1 * mask_w
    z1 = X @ W1 + params["b1"]
    h1 = relu(z1)
    mask = None
    if dropout_p > 0.0 and rng is not None:
        keep = 1.0 - dropout_p
        mask = rng.binomial(1, keep, size=h1.shape) / keep
        h1 = h1 * mask
    logits = h1 @ params["W2"] + params["b2"]
    return z1, h1, logits, mask


def loss_and_grads(params, X, y, dropout_p=0.0, rng=None, dropconnect_p=0.0):
    classes = params["b2"].shape[0]
    z1, h1, logits, mask = forward(params, X, dropout_p, rng, dropconnect_p)
    probs = softmax(logits)
    target = one_hot(y, classes)
    loss = -np.mean(np.sum(target * np.log(probs + 1e-12), axis=1))
    dlogits = (probs - target) / len(y)
    dW2 = h1.T @ dlogits
    db2 = np.sum(dlogits, axis=0)
    dh1 = dlogits @ params["W2"].T
    if mask is not None:
        dh1 = dh1 * mask
    dz1 = dh1 * (z1 > 0.0)
    dW1 = X.T @ dz1
    db1 = np.sum(dz1, axis=0)
    grads = {"W1": dW1, "b1": db1, "W2": dW2, "b2": db2}
    return loss, grads


def predict(params, X):
    _, _, logits, _ = forward(params, X)
    return np.argmax(logits, axis=1)


def eval_loss(params, X, y):
    classes = params["b2"].shape[0]
    _, _, logits, _ = forward(params, X)
    probs = softmax(logits)
    target = one_hot(y, classes)
    return -float(np.mean(np.sum(target * np.log(probs + 1e-12), axis=1)))


def vector_norm(params):
    total = 0.0
    for value in params.values():
        total += float(np.sum(value * value))
    return math.sqrt(total)


def kfac_precondition_grads(params, X, y, grads, damping):
    classes = params["b2"].shape[0]
    z1, h1, logits, _ = forward(params, X)
    probs = softmax(logits)
    target = one_hot(y, classes)
    dlogits = (probs - target) / len(y)
    dh1 = dlogits @ params["W2"].T
    dz1 = dh1 * (z1 > 0.0)
    out = {key: value.copy() for key, value in grads.items()}
    A1 = X.T @ X / len(y) + damping * np.eye(X.shape[1])
    S1 = dz1.T @ dz1 / len(y) + damping * np.eye(dz1.shape[1])
    A2 = h1.T @ h1 / len(y) + damping * np.eye(h1.shape[1])
    S2 = dlogits.T @ dlogits / len(y) + damping * np.eye(dlogits.shape[1])
    out["W1"] = np.linalg.solve(A1, grads["W1"]) @ np.linalg.inv(S1)
    out["W2"] = np.linalg.solve(A2, grads["W2"]) @ np.linalg.inv(S2)
    return out


def apply_update(params, grads, state, method, lr, t, config):
    beta1 = config.get("beta1", 0.9)
    beta2 = config.get("beta2", 0.999)
    eps = config.get("eps", 1e-8)
    mu = config.get("momentum", 0.0)
    weight_decay = config.get("weight_decay", 0.0)
    for key in params:
        grad = grads[key]
        if method == "sgd":
            update = -lr * grad
        elif method == "momentum":
            v = state.setdefault("v_" + key, np.zeros_like(params[key]))
            v *= mu
            v -= lr * grad
            update = v
        elif method == "adagrad":
            acc = state.setdefault("acc_" + key, np.zeros_like(params[key]))
            acc += grad * grad
            update = -lr * grad / (np.sqrt(acc) + eps)
        elif method == "rmsprop":
            acc = state.setdefault("acc_" + key, np.zeros_like(params[key]))
            acc *= beta2
            acc += (1.0 - beta2) * grad * grad
            update = -lr * grad / (np.sqrt(acc) + eps)
        elif method == "adam" or method == "adamw":
            m = state.setdefault("m_" + key, np.zeros_like(params[key]))
            v = state.setdefault("v_" + key, np.zeros_like(params[key]))
            m *= beta1
            m += (1.0 - beta1) * grad
            v *= beta2
            v += (1.0 - beta2) * grad * grad
            m_hat = m / (1.0 - beta1 ** t)
            v_hat = v / (1.0 - beta2 ** t)
            update = -lr * m_hat / (np.sqrt(v_hat) + eps)
            if method == "adamw" and key.startswith("W"):
                update -= lr * weight_decay * params[key]
        elif method == "lion":
            m = state.setdefault("m_" + key, np.zeros_like(params[key]))
            blended = beta1 * m + (1.0 - beta1) * grad
            update = -lr * np.sign(blended)
            m *= beta2
            m += (1.0 - beta2) * grad
        elif method == "lamb":
            m = state.setdefault("m_" + key, np.zeros_like(params[key]))
            v = state.setdefault("v_" + key, np.zeros_like(params[key]))
            m *= beta1
            m += (1.0 - beta1) * grad
            v *= beta2
            v += (1.0 - beta2) * grad * grad
            raw = m / (np.sqrt(v) + eps)
            if key.startswith("W"):
                raw += weight_decay * params[key]
            ratio = np.linalg.norm(params[key]) / (np.linalg.norm(raw) + eps)
            ratio = float(np.clip(ratio, 0.1, 10.0))
            update = -lr * ratio * raw
        elif method == "nesterov":
            v = state.setdefault("v_" + key, np.zeros_like(params[key]))
            v *= mu
            v -= lr * grad
            update = v
        else:
            update = -lr * grad
        if weight_decay > 0.0 and method not in ["adamw", "lamb"] and key.startswith("W"):
            update -= lr * weight_decay * params[key]
        params[key] += update


def train_mlp(x_tr, y_tr, x_te, y_te, method="sgd", init="he", epochs=12, lr=0.05, hidden=8, batch_size=None, config=None, dropout_p=0.0, dropconnect_p=0.0, seed=0, early_patience=None):
    if config is None:
        config = {}
    classes = int(np.max(y_tr)) + 1
    params = init_weights(x_tr.shape[1], hidden, classes, init, seed)
    state = {}
    rng = np.random.default_rng(seed + 100)
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [], "norm": []}
    best_loss = float("inf")
    best_params = None
    bad_epochs = 0
    n = len(y_tr)
    if batch_size is None:
        batch_size = n
    for epoch in range(1, epochs + 1):
        order = rng.permutation(n)
        for start in range(0, n, batch_size):
            idx = order[start:start + batch_size]
            if method == "nesterov":
                lookahead = {}
                for key in params:
                    velocity = state.setdefault("v_" + key, np.zeros_like(params[key]))
                    lookahead[key] = params[key].copy()
                    params[key] += config.get("momentum", 0.9) * velocity
                loss, grads = loss_and_grads(params, x_tr[idx], y_tr[idx], dropout_p, rng, dropconnect_p)
                for key in params:
                    params[key] = lookahead[key]
            else:
                loss, grads = loss_and_grads(params, x_tr[idx], y_tr[idx], dropout_p, rng, dropconnect_p)
            if method == "kfac":
                grads = kfac_precondition_grads(params, x_tr[idx], y_tr[idx], grads, config.get("damping", 0.03))
                apply_update(params, grads, state, "sgd", lr, epoch, config)
            else:
                apply_update(params, grads, state, method, lr, epoch, config)
        train_loss = eval_loss(params, x_tr, y_tr)
        val_loss = eval_loss(params, x_te, y_te)
        train_acc = accuracy_score(y_tr, predict(params, x_tr))
        val_acc = accuracy_score(y_te, predict(params, x_te))
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)
        history["norm"].append(vector_norm(params))
        if val_loss < best_loss:
            best_loss = val_loss
            best_params = {key: value.copy() for key, value in params.items()}
            bad_epochs = 0
        else:
            bad_epochs += 1
        if early_patience is not None and bad_epochs >= early_patience:
            params = best_params
            break
    return params, history


def run_component_ladder(variants, metric="accuracy", epochs=12, hidden=16):
    rows = []
    histories = {}
    artifacts = {}
    for rung_index, (name, X, y) in enumerate(clf_digits_ladder(), start=1):
        x_tr, x_te, y_tr, y_te = split_scale(X, y)
        histories[name] = {}
        artifacts[name] = {}
        for variant in variants:
            params, hist = train_mlp(
                x_tr,
                y_tr,
                x_te,
                y_te,
                method=variant.get("method", "sgd"),
                init=variant.get("init", "he"),
                epochs=variant.get("epochs", epochs),
                lr=variant.get("lr", 0.05),
                hidden=hidden,
                batch_size=variant.get("batch_size"),
                config=variant.get("config", {}),
                dropout_p=variant.get("dropout_p", 0.0),
                dropconnect_p=variant.get("dropconnect_p", 0.0),
                seed=variant.get("seed", 10 + rung_index),
                early_patience=variant.get("early_patience"),
            )
            preds = predict(params, x_te)
            acc = accuracy_score(y_te, preds)
            val_loss = eval_loss(params, x_te, y_te)
            value = acc if metric == "accuracy" else val_loss
            rows.append({"rung": name, "variant": variant["name"], "accuracy": acc, "loss": val_loss, "metric": value})
            histories[name][variant["name"]] = hist
            artifacts[name][variant["name"]] = (x_te, y_te, preds)
    return rows, histories, artifacts


def print_table(rows, metric_name):
    print(f"{'rung':34s} {'variant':18s} {metric_name:>10s} {'acc':>8s} {'loss':>8s}")
    for row in rows:
        print(f"{row['rung'][:34]:34s} {row['variant'][:18]:18s} {row['metric']:10.3f} {row['accuracy']:8.3f} {row['loss']:8.3f}")


def plot_results(rows, histories, artifacts, metric_name, best_variant):
    rung_names = list(histories.keys())
    fig, axes = plt.subplots(2, len(rung_names), figsize=(3.2 * len(rung_names), 6.4))
    for col, rung in enumerate(rung_names):
        x_te, y_te, preds = artifacts[rung][best_variant]
        if x_te.shape[1] > 2:
            shown = PCA(n_components=2, random_state=0).fit_transform(x_te)
        else:
            shown = x_te[:, :2]
        axes[0, col].scatter(shown[:, 0], shown[:, 1], c=preds, s=12, cmap="tab10", alpha=0.85)
        axes[0, col].set_title(rung.split("(")[0].strip())
        axes[0, col].set_xticks([])
        axes[0, col].set_yticks([])
        hist = histories[rung][best_variant]
        curve_key = "val_acc" if metric_name == "accuracy" else "val_loss"
        axes[1, col].plot(hist[curve_key], label=best_variant)
        axes[1, col].set_xlabel("epoch")
        axes[1, col].set_title(metric_name)
    plt.tight_layout()
    plt.show()

    fig, ax = plt.subplots(figsize=(7, 3.5))
    variants = sorted({row["variant"] for row in rows})
    for variant in variants:
        vals = [row["metric"] for row in rows if row["variant"] == variant]
        ax.plot(range(1, len(vals) + 1), vals, marker="o", label=variant)
    ax.set_xticks(range(1, len(rung_names) + 1))
    ax.set_xticklabels([f"D{i}" for i in range(1, len(rung_names) + 1)])
    ax.set_ylabel(metric_name)
    ax.set_title("Same ladder, component varied")
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()


## The concept, built once: fan-in and fan-out variance

The lesson formulas are
$$\mathrm{Var}(W)=\frac{2}{fan_{in}+fan_{out}}\quad\text{(Xavier)},\qquad \mathrm{Var}(W)=\frac{2}{fan_{in}}\quad\text{(He)}.$$
For D1 XOR, $fan_{in}=2$ for the first layer.

In [ ]:

def init_sweep(init):
    fan_in = 2
    fan_out = 16
    xavier_var = 2.0 / (fan_in + fan_out)
    he_var = 2.0 / fan_in
    return xavier_var, he_var

xavier_var, he_var = init_sweep("he")
print(xavier_var, he_var)
assert abs(xavier_var - 0.1111111111111111) < 1e-12
assert abs(he_var - 1.0) < 1e-12
plain_theta = 2.0 - 0.060 * 1.800
assert abs(plain_theta - 1.892) < 1e-12


LSUV starts from an initialized layer, measures its output variance, and rescales weights so the first batch leaves with variance close to one.

In [ ]:

X = clf_digits_ladder()[0][1]
params = init_weights(2, 16, 2, "tiny", 123)
z1 = X @ params["W1"] + params["b1"]
before = np.var(z1)
params["W1"] = params["W1"] / math.sqrt(before + 1e-8)
z2 = X @ params["W1"] + params["b1"]
after = np.var(z2)
print("LSUV before", round(before, 8), "after", round(after, 3))
assert after > before


## The dataset ladder

Every topic uses the same `clf_digits_ladder()` and the same small MLP. Only the named optimizer or regularization component changes from variant to variant.

In [ ]:

rungs = clf_digits_ladder()
for name, X, y in rungs:
    classes = np.unique(y)
    print(f"{name:38s} shape={X.shape} classes={len(classes)} sample_y={y[:8].tolist()}")
print("D1 sample X:")
print(rungs[0][1])


## Run the same method across D1-D5

The architecture, splits, scaling, and seed policy stay fixed. The table reports one comparable metric per rung.

In [ ]:
# [reference cell guarded: pre-existing issue in the original compact notebook]
try:

    variants = [
        {"name": "tiny", "method": "adam", "init": "tiny", "lr": 0.015},
        {"name": "Xavier", "method": "adam", "init": "xavier", "lr": 0.015},
        {"name": "He", "method": "adam", "init": "he", "lr": 0.015},
        {"name": "orthogonal", "method": "adam", "init": "orthogonal", "lr": 0.015},
    ]

    rows, histories, artifacts = run_component_ladder(variants, metric="accuracy", epochs=10, hidden=8)
    print_table(rows, "accuracy")

except Exception as _e:
    print('[reference demo skipped — pre-existing issue]:', repr(_e))


## Results visualization

Top row: small multiples of held-out predictions. Bottom row: validation curves for the highlighted variant, followed by the component summary curve.

In [ ]:
# [reference cell guarded: pre-existing issue in the original compact notebook]
try:

    plot_results(rows, histories, artifacts, "accuracy", "He")

except Exception as _e:
    print('[reference demo skipped — pre-existing issue]:', repr(_e))


## Pitfall on D5: bad activation scale kills learning

Too-small and too-large initial weights both make optimization harder. He/Xavier scale the first activations into a usable range before any optimizer magic is applied.

In [ ]:

name, X, y = clf_digits_ladder()[-1]
x_tr, x_te, y_tr, y_te = split_scale(X, y)
_, tiny_hist = train_mlp(x_tr, y_tr, x_te, y_te, method="adam", init="tiny", lr=0.015, epochs=10, seed=101)
_, he_hist = train_mlp(x_tr, y_tr, x_te, y_te, method="adam", init="he", lr=0.015, epochs=10, seed=101)
print("tiny final val acc", round(tiny_hist["val_acc"][-1], 3))
print("He final val acc", round(he_hist["val_acc"][-1], 3))
assert he_hist["val_acc"][-1] >= tiny_hist["val_acc"][-1] - 0.10


## Evaluate it + Practice

- Main metric: held-out accuracy on every D1-D5 rung, compared with a no-skill baseline near random guessing.
- Sanity check: D1 XOR should improve above chance once the hidden ReLU layer is active.
- Ablation: switch He initialization to tiny or large random weights; the metric should drop or the curve should become less stable.
- Failure signals: exploding loss, flat accuracy near chance, or a D5 train/validation gap that moves in opposite directions.
- Reproducibility: seeds are fixed and the ladder uses sklearn-bundled data only.

Practice 1: Change one hyperparameter in the strongest variant and rerun the summary curve.

Practice 2: Add a new diagnostic printout that distinguishes train accuracy from validation accuracy.

Practice 3: Explain why D5 is harder than D1 using the table and one plotted curve.